# Akan (Asante Twi) Text-to-Speech — VITS fine-tuning
End-to-end pipeline: data prep → fine-tune → inference → evaluation. **Runtime ▸ Change runtime type ▸ GPU (T4 or better).**
Edit the `CONFIG` cell, then run top to bottom. Repo: replace `REPO_URL` with your GitHub URL.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')   # checkpoints are saved to Drive so a Colab disconnect does not lose training

In [ ]:
REPO_URL = "https://github.com/As-a-re/akan-tts"   # <- change
!git clone $REPO_URL /content/akan-tts || true
%cd /content/akan-tts
!pip -q install -r requirements.txt

## 1. Data — choose ONE speaker from an openly licensed corpus
Fill in the dataset id you verified (licence + speaker). Either HF datasets or a local TSV.

In [ ]:
CONFIG = dict(
    hf_dataset="google/WaxalNLP",     # WAXAL (Google), CC-BY-4.0 -- confirm licence on the dataset card
    hf_config="twi_tts",              # Akan/Twi studio TTS subset (~1.1k utterances)
    split="train", audio_col="audio", text_col="transcription",
    speaker_col=None, speaker_id=None,   # TTS subset should be single-speaker; check stats below
    max_hours=None,
)
OUT = "/content/drive/MyDrive/akan_tts"
import os; os.makedirs(OUT, exist_ok=True)
from huggingface_hub import notebook_login; notebook_login()   # needed if the dataset asks you to accept terms

In [ ]:
# If the dataset is gated: from huggingface_hub import login; login()
args = f"--hf_dataset {CONFIG['hf_dataset']} --split {CONFIG['split']} --audio_col {CONFIG['audio_col']} --text_col {CONFIG['text_col']} --out data/akan_ljs"
if CONFIG['hf_config']: args += f" --hf_config {CONFIG['hf_config']}"
if CONFIG['speaker_col']: args += f" --speaker_col {CONFIG['speaker_col']} --speaker_id {CONFIG['speaker_id']}"
if CONFIG['max_hours']: args += f" --max_hours {CONFIG['max_hours']}"
!python scripts/prepare_data.py $args

## 2. Fine-tune VITS (pretrained English LJSpeech checkpoint → Akan graphemes)

In [ ]:
!python scripts/train.py --data data/akan_ljs --out $OUT/runs --epochs 300 --batch_size 16
# If Colab disconnects:  !python scripts/train.py --data data/akan_ljs --out $OUT/runs --continue_path $OUT/runs/<run_folder>

## 3. Inference on unseen text

In [ ]:
import glob
run = sorted(glob.glob(f"{OUT}/runs/akan_vits*"))[-1]
CKPT, CFG = f"{run}/best_model.pth", f"{run}/config.json"
!python scripts/infer.py --ckpt $CKPT --config $CFG --file data/test_sentences.txt --out samples --cuda
from IPython.display import Audio, display
for l in open("samples/prompts.tsv", encoding="utf-8"):
    fn, raw, norm = l.rstrip().split("\t"); print(raw); display(Audio(f"samples/{fn}"))

## 4. Evaluation
ASR round-trip (WER/CER) + a MOS sheet to circulate to Twi-speaking listeners.

In [ ]:
!python scripts/evaluate.py asr --samples samples --out eval/asr_results.csv
!python scripts/evaluate.py mos --samples samples --out eval/mos_sheet.csv

In [ ]:
!cp -r samples eval $OUT/ ; !cp data/akan_ljs/stats.json $OUT/